In [ ]:
#1 Imports and Warnings Suppression

In [ ]:
import requests
from bs4 import BeautifulSoup
from click import prompt
from itext2kg import iText2KG_Star
from newspaper import Article
from youtube_transcript_api import YouTubeTranscriptApi
import yaml
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.caches import BaseCache
from pydantic import BaseModel, Field, ValidationError
from langchain_core.callbacks.base import Callbacks
from typing import List
from iText2KG.itext2kg import iText2KG
import networkx as nx
import matplotlib as plt
from langchain.text_splitter import RecursiveCharacterTextSplitter
import time
import warnings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from neo4j_client import Neo4jClient
import traceback
import json
import os
import re
import asyncio
from enum import Enum
from itext2kg.models.schemas import Facts
from typing import List, Dict, Any, Optional
from itext2kg import iText2KG_Star
from itext2kg.logging_config import setup_logging, get_logger
from langchain_core.caches import BaseCache
# Suppress Pydantic schema warning
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")

ChatOpenAI.BaseCache = BaseCache
ChatOpenAI.model_rebuild()
ChatOllama.BaseCache = BaseCache
ChatOllama.model_rebuild()


In [ ]:
#2 Article Scraping Function 

In [ ]:
def scrape_article(url):
    print(f"Scraping article from: {url}")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/114.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        # Method 1: BeautifulSoup approach
        soup = BeautifulSoup(response.text, 'html.parser')
        content_div = soup.find("div", class_="entry-content ap-pattern--entry-content")

        if content_div:
            article_text = content_div.get_text(separator="\n", strip=True)
            with open("output/pure_scrum_clean.txt", "w", encoding="utf-8") as f:
                f.write(article_text)
            print("Article scraped successfully with BeautifulSoup")
        else:
            print("Specific div not found, trying newspaper3k...")

        # Method 2: Newspaper3k approach (as backup)
        article = Article(url)
        article.download()
        article.parse()

        if article.text:
            with open("output/pure_scrum_clean2.txt", "w", encoding="utf-8") as f:
                f.write(article.text)
            print("Article scraped successfully with newspaper3k")
            return article.text
        else:
            print("Failed to extract article content")
            return None

    except requests.RequestException as e:
        print(f"Error fetching article: {e}")
        return None

In [ ]:
#3. YouTube Transcript Retrieval Function

In [ ]:
def get_youtube_transcript(video_id):
    print(f"Getting YouTube transcript for video: {video_id}")

    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        transcript = " ".join([item['text'] for item in transcript_list])

        with open("output/youtubeVideo.txt", "w", encoding="utf-8") as f:
            f.write(transcript)

        print("YouTube transcript downloaded successfully")
        return transcript

    except Exception as e:
        print(f"Error getting YouTube transcript: {e}")
        return None

In [ ]:
#4. Configuration Loading

In [ ]:
def load_config(config_path="config.yml"):
    try:
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)
        return config
    except FileNotFoundError:
        print("config.yml not found. Creating sample config for Ollama...")

        sample_config = {
            'llm': {
                'model_type': 'ollama',
                'model_name': 'llama3.1',
                'format': 'json',
                'temperature': 0.1,
                'max_tokens': 2000,
                'timeout': 60,
                'max_retries': 3
            },
            'paths': {
                'output_dir': './output'
            }
        }

        with open(config_path, "w") as f:
            yaml.dump(sample_config, f, default_flow_style=False)

        print("Sample config.yml created. Update it if needed.")
        return sample_config


In [ ]:
#5. LLM and Embeddings Initialization Ollama

In [ ]:

def initialize_llm(config):
    if not config:
        return None, None

    llm_config = config.get('llm', {})

    try:
        if llm_config.get('model_type') == "ollama":
            llm = ChatOllama(
                model=llm_config.get('model_name', 'llama3.1'),
                temperature=llm_config.get('temperature', 0.1)
            )

            embeddings = OllamaEmbeddings(model=llm_config.get('model_name', 'llama3.1'))

            print("LLM and Embeddings initialized successfully (Ollama)")
            return llm, embeddings
        else:
            print("Only 'ollama' model type is supported in this setup")
            return None, None
    except Exception as e:
        print(f"Error initializing LLM: {e}")
        return None, None


In [ ]:
#8. Save Triples to File and also filter duplicates

In [ ]:
# # def normalize_triples(triples):
# #     return [(s.strip().lower(), p.strip().lower(), o.strip().lower()) for s, p, o in triples]
# # 
# # 
# # def filter_triples(triples):
# #     filtered = []
# #     for subj, pred, obj in triples:
# #         if subj and pred and obj and subj.lower() != 'none' and obj.lower() != 'none':
# #             filtered.append((subj, pred, obj))
# #     return filtered
# # 
# # 
# # def remove_duplicates(triples):
# #     seen = set()
# #     unique_triples = []
# #     for triple in triples:
# #         if triple not in seen:
# #             unique_triples.append(triple)
# #             seen.add(triple)
# #     return unique_triples
# # 
# # 
def save_triples(triples, filename="extracted_triples.txt"):
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write("Extracted Knowledge Graph Triples:\n")
            f.write("=" * 40 + "\n\n")
            for i, (subject, predicate, obj) in enumerate(triples, 1):
                f.write(f"{i}. ({subject}) --[{predicate}]--> ({obj})\n")
        print(f"Triples saved to {filename}")
    except Exception as e:
        print(f"Error saving triples: {e}")


# 
# def normalize_triples(triples):
#     normalized = []
#     for triple in triples:
#         if len(triple) == 3:
#             subject, predicate, obj = triple
#             subject = str(subject).strip().lower()
#             predicate = str(predicate).strip().lower().replace('_', ' ')
#             obj = str(obj).strip().lower()
#             normalized.append((subject, predicate, obj))
#     return normalized
# 
# 
# def filter_triples(triples):
#     filtered = []
#     for triple in triples:
#         if len(triple) == 3:
#             subject, predicate, obj = triple
#             if len(subject) > 1 and len(predicate) > 1 and len(obj) > 1:
#                 if subject != obj:
#                     filtered.append(triple)
#     return filtered
# 
# 
# def remove_duplicates(triples):
#     return list(set(triples))
# 
# def semantic_filter(triples):
#     hallucinated_terms = {"family", "romantic", "divorce"}
#     return [
#         t for t in triples
#         if not any(term in str(t).lower() for term in hallucinated_terms)
#     ]
# 
# # def save_triples(triples, filename="extracted_triples.txt"):
# #     with open(filename, 'w', encoding='utf-8') as f:
# #         for subject, predicate, obj in triples:
# #             f.write(f"{subject}\t{predicate}\t{obj}\n")


In [ ]:
#11. Plots

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt


def plot_triples(triples):
    G = nx.DiGraph()
    for subj, pred, obj in triples:
        G.add_node(subj)
        G.add_node(obj)
        G.add_edge(subj, obj, label=pred)

    pos = nx.spring_layout(G, k=0.5, iterations=50)

    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=3000, font_size=10, arrowsize=20)

    edge_labels = {(subj, obj): pred for subj, pred, obj in triples}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=9)

    plt.title("Knowledge Graph Visualization")
    plt.show()


In [ ]:
#Custom SCHEMA

In [ ]:
class ScrumEntityType(str, Enum):
    # Core Scrum Roles
    PRODUCT_OWNER = "product_owner"
    SCRUM_MASTER = "scrum_master" 
    DEVELOPER = "developer"
    SCRUM_TEAM = "scrum_team"
    STAKEHOLDER = "stakeholder"
    
    # Scrum Artifacts
    PRODUCT_BACKLOG = "product_backlog"
    SPRINT_BACKLOG = "sprint_backlog"
    INCREMENT = "increment"
    PRODUCT_GOAL = "product_goal"
    SPRINT_GOAL = "sprint_goal"
    
    # Scrum Events
    SPRINT = "sprint"
    SPRINT_PLANNING = "sprint_planning"
    DAILY_SCRUM = "daily_scrum"
    SPRINT_REVIEW = "sprint_review"
    SPRINT_RETROSPECTIVE = "sprint_retrospective"
    
    # Other Concepts - EXPANDED
    PROCESS = "process"
    SYSTEM = "system"
    REQUIREMENT = "requirement"
    USER_STORY = "user_story"
    FEATURE = "feature"
    BUG = "bug"
    TASK = "task"
    ACCEPTANCE_CRITERIA = "acceptance_criteria"
    TECHNICAL_DEBT = "technical_debt"
    DEFINITION_OF_DONE = "definition_of_done"
    
    # General categories for flexibility
    PERSON = "person"
    ROLE = "role"
    ARTIFACT = "artifact"
    EVENT = "event"
    CONCEPT = "concept"
    
    # Fallback values
    UNKNOWN = "unknown"
    NONE = "none"

class ScrumRelationType(str, Enum):
    # Role relationships
    MANAGES = "manages"
    OWNS = "owns"
    DEVELOPS = "develops"
    COACHES = "coaches"
    FACILITATES = "facilitates"
    PARTICIPATES_IN = "participates_in"
    LEADS = "leads"
    
    # Responsibility relationships - EXPANDED
    RESPONSIBLE_FOR = "responsible_for"
    ACCOUNTABLE_FOR = "accountable_for"
    OVERSEES = "oversees"
    MAINTAINS = "maintains"
    ENSURES = "ensures"
    
    # Artifact relationships
    CONTAINS = "contains"
    PRODUCES = "produces"
    REFINES = "refines"
    DEFINES = "defines"
    CONTRIBUTES_TO = "contributes_to"
    INCLUDES = "includes"
    CONSISTS_OF = "consists_of"
    COMPRISES = "comprises"
    
    # Action relationships - EXPANDED
    CREATES = "creates"
    BUILDS = "builds"
    DELIVERS = "delivers"
    PROVIDES = "provides"
    DEVELOPS = "develops"
    
    # Event relationships
    CONDUCTS = "conducts"
    ATTENDS = "attends"
    REVIEWS = "reviews"
    PLANS = "plans"
    RUNS = "runs"
    HOSTS = "hosts"
    ORGANIZES = "organizes"
    
    # Collaboration relationships - EXPANDED
    WORKS_WITH = "works_with"
    COLLABORATES_WITH = "collaborates_with"
    COMMUNICATES_WITH = "communicates_with"
    COORDINATES = "coordinates"
    SUPPORTS = "supports"
    HELPS = "helps"
    
    # Quality relationships - EXPANDED
    INSPECTS = "inspects"
    EVALUATES = "evaluates"
    SPECIFIES = "specifies"
    PRIORITIZES = "prioritizes"
    IMPROVES = "improves"
    OPTIMIZES = "optimizes"
    
    # General relationships
    RELATES_TO = "relates_to"
    IS = "is"
    HAS = "has"
    ACTS_AS = "acts_as"
    
    # Fallback values
    UNKNOWN = "unknown"
    NONE = "none"

# ============ MORE FLEXIBLE SCRUM FACT ============
class ScrumFact(BaseModel):
    statement: str = Field(description="Complete factual statement about Scrum context")
    subject: str = Field(description="Main subject (person, role, artifact, event)")
    predicate: str = Field(description="Action or relationship type") 
    object: str = Field(description="Target object of the relationship")
    
    subject_type: Optional[ScrumEntityType] = Field(default=ScrumEntityType.UNKNOWN, description="Type of subject entity")
    object_type: Optional[ScrumEntityType] = Field(default=ScrumEntityType.UNKNOWN, description="Type of object entity")
    relation_type: Optional[ScrumRelationType] = Field(default=ScrumRelationType.RELATES_TO, description="Standardized relationship type")
    
    confidence: float = Field(default=0.8, ge=0, le=1, description="Extraction confidence score")
    scrum_context: str = Field(default="General", description="Which Scrum concept this relates to")

class ScrumProjectFacts(BaseModel):
    facts: List[ScrumFact] = Field(default=[], description="Structured Scrum project facts")

# ============ SIMPLIFIED IE QUERY ============
def get_scrum_IE_query():
    return '''
    # You are a Scrum expert analyzing project documentation.
    # Extract facts about WHO does WHAT in the Scrum context.
    
    # FOCUS ON:
    - People and their roles (Pat, Product Owner, Scrum Master, Developers)
    - What they do (manages, creates, facilitates, reviews, owns)
    - What they work with (Product Backlog, Sprint Planning, Stakeholders)
    - Relationships between people and artifacts/events
    
    # EXTRACT FACTS AS:
    - Subject: WHO (Pat, Product Owner, Team, etc.)
    - Predicate: ACTION (manages, creates, facilitates, is responsible for, etc.) 
    - Object: WHAT (Product Backlog, Sprint Goal, Daily Scrum, etc.)
    - Statement: Complete sentence describing the relationship
    
    # EXAMPLES:
    "Pat manages the product backlog" → 
    Subject: "Pat", Predicate: "manages", Object: "Product Backlog"
    
    "The Product Owner is responsible for maximizing product value" →
    Subject: "Product Owner", Predicate: "responsible for", Object: "maximizing product value"
    
    "Scrum Master facilitates Sprint Planning" →
    Subject: "Scrum Master", Predicate: "facilitates", Object: "Sprint Planning"
    
    "Developers create working increments" →
    Subject: "Developers", Predicate: "create", Object: "working increments"
    
    # Be flexible with predicates - use natural language, don't force into rigid categories.
    # If unsure about types, leave as "unknown" - focus on getting good subject-predicate-object triples.
    '''

In [ ]:
#Main

In [ ]:
async def main(input_source: str = "text.txt"):
    print("Starting Enhanced Knowledge Graph Extraction with Scrum Schema + iText2KG_Star\n")

    # ============ LOAD INPUT TEXTS ============
    texts = []
    try:
        if isinstance(input_source, str) and os.path.exists(input_source):
            print(f"Loading from file: {input_source}")
            with open(input_source, "r", encoding="utf-8") as f:
                content = f.read().strip()
    
                if len(content) > 1000:
                    sentences = content.split('. ')
                    current_chunk = ""
                    for sentence in sentences:
                        if len(current_chunk + sentence) <= 1000:
                            current_chunk += sentence + ". "
                        else:
                            if current_chunk:
                                texts.append(current_chunk.strip())
                            current_chunk = sentence + ". "
                    if current_chunk:
                        texts.append(current_chunk.strip())
                else:
                    texts = [content]
    
        elif isinstance(input_source, str) and len(input_source.strip()) > 10:
            print("Using direct text input")
            texts = [input_source.strip()]
    
        else:
            print(f"Invalid input source: {input_source}")
            return []
    
    except Exception as e:
        print(f"Error loading input: {e}")
        return []
    
    texts = [text for text in texts if len(text.strip()) > 10]
    
    if not texts:
        print("No valid texts found. Exiting.")
        return []
    
    print(f"Loaded {len(texts)} text segments")
    for i, text in enumerate(texts[:3]):
        print(f"   Text {i + 1} preview: {text[:150]}...")
    
    # ============ INITIALIZE LLM ============
    print("\nInitializing LLM and Embeddings...")
    config = load_config()
    llm, embeddings = initialize_llm(config)
    
    if not llm or not embeddings:
        print("Cannot proceed without LLM initialization")
        return []
    
    print(f"LLM and Embeddings initialized successfully")
    print(f"LLM initialized: {config['llm']['model_name']}")

    # ============ COMPREHENSIVE EXTRACTION ============
    print("\nExtracting knowledge with multiple methods...")
    
    all_triples = []
    
    # METHOD 1: iText2KG_Star with better parameters
    print("Trying iText2KG_Star...")
    try:
        from itext2kg import iText2KG_Star
        itext2kg_star = iText2KG_Star(llm_model=llm, embeddings_model=embeddings)
        
        kg_result = await itext2kg_star.build_graph(
            sections=texts,
            ent_threshold=0.4, 
            rel_threshold=0.4,  
            max_tries=3,
            entity_name_weight=0.7,
            entity_label_weight=0.3,
            observation_date="2025-08-10"
        )
        
        if kg_result and hasattr(kg_result, 'relationships'):
            for rel in kg_result.relationships:
                try:
                    subject = None
                    if hasattr(rel, 'startEntity'):
                        if hasattr(rel.startEntity, 'name') and rel.startEntity.name:
                            subject = str(rel.startEntity.name).strip()
                        elif hasattr(rel.startEntity, 'label') and rel.startEntity.label:
                            subject = str(rel.startEntity.label).strip()
                    
                    obj = None
                    if hasattr(rel, 'endEntity'):
                        if hasattr(rel.endEntity, 'name') and rel.endEntity.name:
                            obj = str(rel.endEntity.name).strip()
                        elif hasattr(rel.endEntity, 'label') and rel.endEntity.label:
                            obj = str(rel.endEntity.label).strip()
                    
                    predicate = "relates_to"
                    if hasattr(rel, 'name') and rel.name:
                        predicate = str(rel.name).strip().lower().replace(' ', '_')
                    
                    if subject and obj and subject != obj:
                        subject = re.sub(r'\s+', ' ', subject.strip()).title()
                        obj = re.sub(r'\s+', ' ', obj.strip()).title()
                        predicate = re.sub(r'[^\w_]', '', predicate.lower())
                        
                        if len(subject) > 1 and len(obj) > 1 and len(obj) < 100:
                            all_triples.append((subject, predicate, obj))
                except:
                    continue
            
            print(f"iText2KG_Star found {len(all_triples)} triples")
        else:
            print("iText2KG_Star returned no results")
            
    except Exception as e:
        print(f"iText2KG_Star failed: {e}")
    
    # METHOD 2: Document Distiller
    print("Trying Document Distiller...")
    try:
        from itext2kg import DocumentsDistiller
        
        document_distiller = DocumentsDistiller(llm_model=llm)
        scrum_IE_query = get_scrum_IE_query()
        
        for text in texts:
            try:
                facts_result = await document_distiller.distill(
                    documents=[text],
                    IE_query=scrum_IE_query,
                    output_data_structure=ScrumProjectFacts
                )
                
                if facts_result and hasattr(facts_result, 'facts'):
                    for fact in facts_result.facts:
                        if (hasattr(fact, 'subject') and hasattr(fact, 'object') and 
                            fact.subject and fact.object):
                            
                            subject = str(fact.subject).strip().title()
                            obj = str(fact.object).strip().title()
                            
                            predicate = "relates_to"
                            if hasattr(fact, 'predicate') and fact.predicate:
                                predicate = str(fact.predicate).strip().lower().replace(' ', '_')
                            elif hasattr(fact, 'relation_type') and fact.relation_type:
                                predicate = str(fact.relation_type).strip().lower().replace(' ', '_')
                            
                            if subject and obj and subject != obj and len(subject) > 1 and len(obj) > 1:
                                all_triples.append((subject, predicate, obj))
                
            except Exception as e:
                continue
        
        print(f"Document Distiller processed")
        
    except Exception as e:
        print(f"Document Distiller failed: {e}")
    
    # METHOD 3: ADAPTIVE PATTERN MATCHING - WORKS WITH ANY TEXT!
    print("Applying adaptive pattern matching...")
    
    universal_patterns = [
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(?:is|are)\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'is'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(?:was|were)\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'was'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+acts?\s+as\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'acts_as'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+serves?\s+as\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'serves_as'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+works?\s+as\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'works_as'),
        
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(?:has|have)\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'has'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+owns?\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'owns'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+contains?\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'contains'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+includes?\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'includes'),
        
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(manages?|creates?|builds?|develops?|leads?|runs?|handles?|maintains?|oversees?|controls?)\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'action_verb'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(provides?|delivers?|gives?|offers?|supplies?|produces?)\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'provides_verb'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(uses?|utilizes?|employs?|applies?)\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'uses'),
        
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(?:is|are)\s+responsible\s+for\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'responsible_for'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(?:is|are)\s+accountable\s+for\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'accountable_for'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(?:is|are)\s+in\s+charge\s+of\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'in_charge_of'),
        
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+works?\s+with\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'works_with'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+collaborates?\s+with\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'collaborates_with'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+partners?\s+with\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'partners_with'),
        
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(participates?|joins?|attends?)\s+(?:in\s+)?(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'participates_in'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(organizes?|hosts?|conducts?|facilitates?)\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'organizes'),
        
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(?:is|are)\s+(?:located\s+)?(?:in|at|on)\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'located_in'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+(?:belongs?|pertains?)\s+to\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'belongs_to'),
        
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+will\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'will'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+can\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'can'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+should\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'should'),
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+must\s+([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'must'),
        
        (r'(\b[A-Z][a-zA-Z\s]+?)\s+([a-zA-Z]+s)\s+(?:a|an|the)?\s*([a-zA-Z][^.!?]{1,80}?)(?:[.!?]|$)', 'general_action'),
    ]
    
    pattern_triples = []
    total_sentences = 0
    
    for text_idx, text in enumerate(texts):
        print(f"Processing text {text_idx + 1}...")
        
        clean_text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
        clean_text = re.sub(r'\s+', ' ', clean_text)
        
        sentences = re.split(r'[.!?]+', clean_text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
        total_sentences += len(sentences)
        
        print(f"  Found {len(sentences)} sentences to process")
        
        for sent_idx, sentence in enumerate(sentences):
            found_in_sentence = 0
            
            for pattern, relation_type in universal_patterns:
                matches = list(re.finditer(pattern, sentence, re.IGNORECASE))
                
                for match in matches:
                    try:
                        if relation_type in ['action_verb', 'provides_verb']:
                            subject = match.group(1).strip()
                            predicate = match.group(2).strip().lower()
                            obj = match.group(3).strip()
                        elif relation_type == 'general_action':
                            subject = match.group(1).strip()
                            predicate = match.group(2).strip().lower()
                            obj = match.group(3).strip()
                        else:
                            subject = match.group(1).strip()
                            predicate = relation_type
                            obj = match.group(2).strip()
                        
                        subject = re.sub(r'\s+', ' ', subject).strip()
                        if subject.lower().startswith('the '):
                            subject = subject[4:]
                        subject = subject.title()
                        
                        obj = re.sub(r'\s+', ' ', obj).strip()
                        if obj.lower().startswith('the '):
                            obj = obj[4:]
                        elif obj.lower().startswith('a '):
                            obj = obj[2:]
                        elif obj.lower().startswith('an '):
                            obj = obj[3:]
                        
                        obj = re.sub(r'[,;:]+$', '', obj).strip()
                        obj = obj.title()
                        
                        predicate = re.sub(r'[^\w_]', '', predicate.lower())
                        if not predicate:
                            predicate = 'relates_to'
                        
                        if (len(subject) > 1 and len(obj) > 1 and len(obj) < 100 and
                            subject.lower() != obj.lower() and
                            obj.lower() not in ['it', 'this', 'that', 'they', 'them', 'he', 'she'] and
                            not obj.lower().startswith('somebody') and
                            predicate not in ['the', 'a', 'an', 'and', 'or']):
                            
                            pattern_triples.append((subject, predicate, obj))
                            found_in_sentence += 1
                            
                    except (IndexError, AttributeError):
                        continue
            
            if found_in_sentence == 0 and sent_idx < 3:
                print(f"No matches in: {sentence[:100]}...")
    
    all_triples.extend(pattern_triples)
    print(f"Pattern matching found {len(pattern_triples)} triples from {total_sentences} sentences")
    
    # METHOD 4: LLM-based extraction for any remaining gaps
    print("Applying LLM-based extraction...")
    try:
        llm_triples = []
        
        for text in texts:
            if len(text) > 50:
                prompt = get_scrum_IE_query()
                
                try:
                    response = await llm.ainvoke(prompt)
                    if hasattr(response, 'content'):
                        response_text = response.content
                    else:
                        response_text = str(response)
                    
                    lines = response_text.split('\n')
                    for line in lines:
                        if '|' in line:
                            parts = [p.strip() for p in line.split('|')]
                            if len(parts) >= 3:
                                subject, predicate, obj = parts[0], parts[1], parts[2]
                                
                                subject = re.sub(r'\s+', ' ', subject).strip().title()
                                predicate = re.sub(r'[^\w_]', '', predicate.lower())
                                obj = re.sub(r'\s+', ' ', obj).strip().title()
                                
                                if (len(subject) > 1 and len(obj) > 1 and len(obj) < 100 and
                                    subject.lower() != obj.lower()):
                                    llm_triples.append((subject, predicate, obj))
                                    
                except Exception as e:
                    continue
        
        all_triples.extend(llm_triples)
        print(f"✓ LLM extraction found {len(llm_triples)} triples")
        
    except Exception as e:
        print(f"✗ LLM extraction failed: {e}")
    
    # ============ SMART POST-PROCESSING ============
    print(f"\nPost-processing {len(all_triples)} triples...")
    
    seen_normalized = {}
    filtered_triples = []
    
    for subject, predicate, obj in all_triples:
        norm_key = (
            subject.lower().strip(),
            predicate.lower().strip(),
            obj.lower().strip()
        )
        
        if norm_key not in seen_normalized:
            if (len(subject.strip()) > 1 and 
                len(obj.strip()) > 1 and 
                len(obj.strip()) < 150 and  
                subject.lower().strip() != obj.lower().strip() and
                obj.lower() not in ['it', 'this', 'that', 'they', 'them'] and
                predicate not in ['the', 'a', 'an'] and
                not any(word in obj.lower() for word in ['somebody', 'someone', 'maybe', 'perhaps'])):
                
                seen_normalized[norm_key] = True
                filtered_triples.append((
                    subject.strip().title(), 
                    predicate.lower(), 
                    obj.strip().title()
                ))
    
    all_triples = filtered_triples
    print(f"After deduplication and filtering: {len(all_triples)} triples")
    
    # ============ RESULTS ============
    if all_triples:
        try:
            with open("extracted_triples.txt", "w", encoding="utf-8") as f:
                for subject, predicate, obj in all_triples:
                    f.write(f"({subject}) --[{predicate}]--> ({obj})\n")
            print(f"Triples saved to scrum_extracted_triples.txt")
        except Exception as e:
            print(f"Could not save triples: {e}")
        
        print("\nEXTRACTED KNOWLEDGE GRAPH:")
        print("=" * 60)
        for i, (subject, predicate, obj) in enumerate(all_triples[:40], 1):
            print(f"  {i:2d}. ({subject}) --[{predicate}]--> ({obj})")
        
        if len(all_triples) > 40:
            print(f"  ... and {len(all_triples) - 40} more triples")
        
        print(f"\nSUCCESS! Extracted {len(all_triples)} knowledge relationships")
        print(f"Methods used: iText2KG_Star, Document Distiller, Pattern Matching, LLM Extraction")
        
        
        
        try:
            plot_triples(all_triples[:25])
        except Exception as e:
            print(f"Visualization skipped: {e}")
        
        return all_triples
    
    else:
        print("\nNo triples were successfully extracted")
        print("This could be due to:")
        print("  - Text doesn't contain clear relationships")
        print("  - Text is too short or fragmented")
        print("  - LLM/embedding models not working properly")
        print("  - Text needs different preprocessing")
        return []


In [ ]:
if __name__ == "__main__":
    import nest_asyncio

    nest_asyncio.apply()


    async def run_star_pipeline():
        triples = await main("text.txt")
        print(f"\nPipeline completed! Found {len(triples)} triples")
        return triples


    loop = asyncio.get_event_loop()
    triples = loop.run_until_complete(run_star_pipeline())
